In [ ]:
# main.py
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, datasets, models
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
from cvae import CVAE
from ddpm import DDPM
from utils import generate_synthetic_images, evaluate_classifier

# --------------------------
# 1. Hyperparameters
# --------------------------
IMG_SIZE = 64
BATCH_SIZE_CVAE = 64
BATCH_SIZE_DDPM = 16
EPOCHS_CVAE = 100
EPOCHS_DDPM = 50
LATENT_DIM = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 15
T_DIFFUSION = 50
LR_CVAE = 1e-3
LR_DDPM = 2e-4

# --------------------------
# 2. Dataset preparation
# --------------------------
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder("data/PlantVillage", transform=transform)
num_train = int(0.7 * len(dataset))
num_val = int(0.15 * len(dataset))
num_test = len(dataset) - num_train - num_val
train_data, val_data, test_data = random_split(dataset, [num_train, num_val, num_test])

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE_CVAE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE_CVAE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE_CVAE, shuffle=False)

# --------------------------
# 3. CVAE model training
# --------------------------
cvae = CVAE(input_channels=3, latent_dim=LATENT_DIM, num_classes=NUM_CLASSES).to(DEVICE)
optimizer_cvae = optim.Adam(cvae.parameters(), lr=LR_CVAE)
criterion_cvae = nn.MSELoss()

for epoch in range(EPOCHS_CVAE):
    cvae.train()
    running_loss = 0
    for imgs, labels in tqdm(train_loader, desc=f"CVAE Epoch {epoch+1}/{EPOCHS_CVAE}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer_cvae.zero_grad()
        recon_imgs, mu, logvar = cvae(imgs, labels)
        recon_loss = criterion_cvae(recon_imgs, imgs)
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        loss = recon_loss + kl_loss
        loss.backward()
        optimizer_cvae.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, CVAE Loss: {running_loss/len(train_loader):.4f}")

# --------------------------
# 4. DDPM model training
# --------------------------
ddpm = DDPM(img_size=IMG_SIZE, channels=3, T=T_DIFFUSION).to(DEVICE)
optimizer_ddpm = optim.Adam(ddpm.parameters(), lr=LR_DDPM)
criterion_ddpm = nn.MSELoss()

for epoch in range(EPOCHS_DDPM):
    ddpm.train()
    running_loss = 0
    for imgs, _ in tqdm(train_loader, desc=f"DDPM Epoch {epoch+1}/{EPOCHS_DDPM}"):
        imgs = imgs.to(DEVICE)
        optimizer_ddpm.zero_grad()
        loss = ddpm(imgs)
        loss.backward()
        optimizer_ddpm.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, DDPM Loss: {running_loss/len(train_loader):.4f}")

# --------------------------
# 5. Generate synthetic images
# --------------------------
num_synth_per_class = 50  # Adjust based on class balance
cvae_imgs, cvae_labels = generate_synthetic_images(cvae, NUM_CLASSES, num_synth_per_class, DEVICE, latent_dim=LATENT_DIM)
ddpm_imgs, ddpm_labels = generate_synthetic_images(ddpm, NUM_CLASSES, num_synth_per_class, DEVICE, diffusion=True)

# Combine with real train set
train_imgs_real = torch.cat([imgs for imgs, _ in train_loader], dim=0)
train_labels_real = torch.cat([labels for _, labels in train_loader], dim=0)
train_imgs = torch.cat([train_imgs_real, cvae_imgs, ddpm_imgs], dim=0)
train_labels = torch.cat([train_labels_real, cvae_labels, ddpm_labels], dim=0)

# --------------------------
# 6. Train ResNet18 classifier
# --------------------------
resnet = models.resnet18(pretrained=False)
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
resnet = resnet.to(DEVICE)
optimizer_cls = optim.Adam(resnet.parameters(), lr=1e-3)
criterion_cls = nn.CrossEntropyLoss()

train_dataset_hybrid = torch.utils.data.TensorDataset(train_imgs, train_labels)
train_loader_hybrid = DataLoader(train_dataset_hybrid, batch_size=32, shuffle=True)

for epoch in range(30):
    resnet.train()
    running_loss = 0
    for imgs, labels in train_loader_hybrid:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer_cls.zero_grad()
        outputs = resnet(imgs)
        loss = criterion_cls(outputs, labels)
        loss.backward()
        optimizer_cls.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Classifier Loss: {running_loss/len(train_loader_hybrid):.4f}")

# --------------------------
# 7. Evaluation
# --------------------------
accuracy, precision, recall, f1, cm = evaluate_classifier(resnet, test_loader, DEVICE, NUM_CLASSES)
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(cm)


# CVAE

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class CVAE(nn.Module):
    def __init__(self, input_channels=3, latent_dim=128, num_classes=15):
        super().__init__()
        self.latent_dim = latent_dim
        self.num_classes = num_classes

        self.encoder_conv = nn.Sequential(
            nn.Conv2d(input_channels, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU()
        )
        self.flatten = nn.Flatten()
        self.fc_mu = nn.Linear(128*8*8 + num_classes, latent_dim)
        self.fc_logvar = nn.Linear(128*8*8 + num_classes, latent_dim)
        self.fc_dec = nn.Linear(latent_dim + num_classes, 128*8*8)

        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, input_channels, 4, 2, 1), nn.Sigmoid()
        )

    def encode(self, x, y):
        y_onehot = F.one_hot(y, num_classes=self.num_classes).float()
        h = self.encoder_conv(x)
        h = self.flatten(h)
        h = torch.cat([h, y_onehot], dim=1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z, y):
        y_onehot = F.one_hot(y, num_classes=self.num_classes).float()
        h = torch.cat([z, y_onehot], dim=1)
        h = self.fc_dec(h)
        h = h.view(-1, 128, 8, 8)
        x_recon = self.decoder_conv(h)
        return x_recon

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z, y)
        return x_recon, mu, logvar


# DDPM

In [ ]:
# ----------------------------
# Imports essentiels
# ----------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision
from torchvision import transforms, datasets, models
from torchvision.utils import save_image
import numpy as np
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
from diffusers import DDPMPipeline

# ----------------------------
# Paramètres globaux
# ----------------------------
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_CVAE = 64
BATCH_DDPM = 16
LATENT_DIM = 128
IMG_SIZE = 64
NUM_CLASSES = 15
EPOCHS_CVAE = 100
EPOCHS_CLASSIFIER = 50
LEARNING_RATE_CVAE = 1e-3
LEARNING_RATE_CLASSIFIER = 2e-4
DATA_PATH = "./plantvillage"  # chemin vers le dataset PlantVillage

# ----------------------------
# Transformations
# ----------------------------
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

# ----------------------------
# Dataset et DataLoader
# ----------------------------
dataset = datasets.ImageFolder(root=DATA_PATH, transform=transform)
total_size = len(dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_CVAE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_CVAE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_CVAE)

# ----------------------------
# CVAE
# ----------------------------
class CVAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, num_classes=NUM_CLASSES, img_channels=3):
        super().__init__()
        self.latent_dim = latent_dim
        self.num_classes = num_classes
        self.img_channels = img_channels

        self.encoder = nn.Sequential(
            nn.Conv2d(img_channels + num_classes, 32, 4, 2, 1),  # 64x64 -> 32x32
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),  # 32x32 -> 16x16
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1),  # 16x16 -> 8x8
            nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(128*8*8, latent_dim)
        self.fc_logvar = nn.Linear(128*8*8, latent_dim)
        self.fc_decode = nn.Linear(latent_dim + num_classes, 128*8*8)

        self.decoder = nn.Sequential(
            nn.Unflatten(1, (128, 8, 8)),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),  # 8x8 -> 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),   # 16x16 -> 32x32
            nn.ReLU(),
            nn.ConvTranspose2d(32, img_channels, 4, 2, 1),  # 32x32 -> 64x64
            nn.Sigmoid()
        )

    def encode(self, x, y_onehot):
        y_map = y_onehot[:, :, None, None].expand(-1, -1, x.size(2), x.size(3))
        x_cond = torch.cat([x, y_map], dim=1)
        h = self.encoder(x_cond)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, y_onehot):
        z_cond = torch.cat([z, y_onehot], dim=1)
        h = self.fc_decode(z_cond)
        x_hat = self.decoder(h)
        return x_hat

    def forward(self, x, y_onehot):
        mu, logvar = self.encode(x, y_onehot)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z, y_onehot)
        return x_hat, mu, logvar

def cvae_loss(x_hat, x, mu, logvar, beta=1.0):
    recon_loss = F.mse_loss(x_hat, x)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon_loss + beta * kl_loss

# ----------------------------
# Entraînement CVAE
# ----------------------------
def train_cvae(model, loader, epochs=EPOCHS_CVAE):
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE_CVAE)
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for imgs, labels in tqdm(loader):
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)
            labels_onehot = F.one_hot(labels, NUM_CLASSES).float()
            optimizer.zero_grad()
            x_hat, mu, logvar = model(imgs, labels_onehot)
            loss = cvae_loss(x_hat, imgs, mu, logvar)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"[CVAE] Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")

# ----------------------------
# DDPM pré-entraîné fine-tuning
# ----------------------------
ddpm_model = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32")  # exemple CIFAR10 pré-entraîné
ddpm_model.to(DEVICE)

# Optionnel : fine-tuning léger sur PlantVillage
# Boucle simplifiée pour quelques epochs seulement
def finetune_ddpm(model, loader, epochs=5):
    optimizer = torch.optim.Adam(model.unet.parameters(), lr=1e-5)
    model.train()
    for epoch in range(epochs):
        for imgs, _ in tqdm(loader):
            imgs = imgs.to(DEVICE)
            # Forward DDPM step : simplifié, on ne refait pas tout le training full
            optimizer.zero_grad()
            # placeholder, fine-tuning symbolique
            loss = torch.tensor(0.0, requires_grad=True, device=DEVICE)
            loss.backward()
            optimizer.step()

# ----------------------------
# Génération d’images synthétiques
# ----------------------------
def generate_cvae_images(model, num_per_class=50):
    model.eval()
    synthetic_imgs = []
    synthetic_labels = []
    for class_idx in range(NUM_CLASSES):
        y_onehot = F.one_hot(torch.tensor([class_idx]*num_per_class), NUM_CLASSES).float().to(DEVICE)
        z = torch.randn(num_per_class, LATENT_DIM).to(DEVICE)
        with torch.no_grad():
            x_gen = model.decode(z, y_onehot)
        synthetic_imgs.append(x_gen.cpu())
        synthetic_labels.extend([class_idx]*num_per_class)
    synthetic_imgs = torch.cat(synthetic_imgs, dim=0)
    return synthetic_imgs, synthetic_labels

def generate_ddpm_images(model, num_images=50):
    model.eval()
    synthetic_imgs = []
    for _ in range(num_images):
        with torch.no_grad():
            img = model(num_inference_steps=50).images[0]
        synthetic_imgs.append(transforms.ToTensor()(img))
    synthetic_imgs = torch.stack(synthetic_imgs)
    return synthetic_imgs

# ----------------------------
# Fusion datasets
# ----------------------------
def create_hybrid_dataset(real_loader, cvae_imgs, cvae_labels, ddpm_imgs):
    # Real images
    real_imgs = []
    real_labels = []
    for imgs, labels in real_loader:
        real_imgs.append(imgs)
        real_labels.extend(labels.numpy())
    real_imgs = torch.cat(real_imgs, dim=0)

    # CVAE images
    cvae_imgs = cvae_imgs
    cvae_labels = np.array(cvae_labels)

    # DDPM images (labels = placeholder, car non conditionnel)
    ddpm_labels = np.random.randint(0, NUM_CLASSES, len(ddpm_imgs))
    ddpm_imgs = ddpm_imgs

    # Fusion
    all_imgs = torch.cat([real_imgs, cvae_imgs, ddpm_imgs], dim=0)
    all_labels = np.concatenate([real_labels, cvae_labels, ddpm_labels], axis=0)
    return all_imgs, all_labels

# ----------------------------
# DataLoader pour classifier
# ----------------------------
class CustomDataset(Dataset):
    def __init__(self, imgs, labels):
        self.imgs = imgs
        self.labels = labels

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        return self.imgs[idx], self.labels[idx]

# ----------------------------
# Classificateur ResNet18
# ----------------------------
resnet = models.resnet18(pretrained=True)
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
resnet = resnet.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer_cls = torch.optim.Adam(resnet.parameters(), lr=LEARNING_RATE_CLASSIFIER)

# ----------------------------
# Entraînement classificateur
# ----------------------------
def train_classifier(model, train_loader, val_loader, epochs=EPOCHS_CLASSIFIER):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for imgs, labels in tqdm(train_loader):
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer_cls.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer_cls.step()
            total_loss += loss.item()
        print(f"[Classifier] Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")
        evaluate(model, val_loader)

# ----------------------------
# Évaluation
# ----------------------------
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1-score: {f1:.4f}")
    return acc, prec, rec, f1

# ----------------------------
# Exemple d'exécution
# ----------------------------
if __name__ == "__main__":
    # 1. Entraîner CVAE
    cvae = CVAE().to(DEVICE)
    train_cvae(cvae, train_loader, epochs=10)  # pour test rapide, sinon 100 epochs

    # 2. Fine-tuning DDPM (optionnel)
    finetune_ddpm(ddpm_model, train_loader, epochs=1)  # symbolique

    # 3. Génération images synthétiques
    cvae_imgs, cvae_labels = generate_cvae_images(cvae, num_per_class=20)
    ddpm_imgs = generate_ddpm_images(ddpm_model, num_images=50)

    # 4. Fusion dataset hybride
    hybrid_imgs, hybrid_labels = create_hybrid_dataset(train_loader, cvae_imgs, cvae_labels, ddpm_imgs)
    hybrid_dataset = CustomDataset(hybrid_imgs, hybrid_labels)
    hybrid_loader = DataLoader(hybrid_dataset, batch_size=BATCH_CVAE, shuffle=True)

    # 5. Entraîner ResNet18
    train_classifier(resnet, hybrid_loader, val_loader, epochs=5)  # pour test rapide

    # 6. Évaluation finale
    print("Évaluation sur le test set réel :")
    evaluate(resnet, test_loader)
